# AI Crypto Trading Lab — Colab (20 USDT → 500 USDT, Futures x500)
> **Thị trường thật — Tiền ảo — Sàn mô phỏng**
>
Notebook này chạy **toàn bộ** trên Colab: lấy lịch sử tất cả coin/tất cả khung giờ, chạy hàng nghìn lượt trade futures và kết luận đạt/tệ.
>
-> **Yêu cầu:** 20 USDT → 500 USDT (x25) với futures tối đa **x500**, dựa trên **tất cả nến** mọi khung + luật trade.
-> **Kết luận sơ bộ trên 1000 nhánh thật (1h 5 coin 36 ngày): chỉ 0.1% đạt 500, x500 liquidation 52%, tốt nhất là x20 hold_long SOL** — xem chi tiết `scripts/evaluate_futures_x500.py`.


## 0) Chuẩn bị — Mount Drive (khuyến nghị) và clone repo
Colab T4 GPU đủ cho lab; với 100k branching/10M steps nên dùng A100. Lưu data vào Drive để không mất khi disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Thay <your-org> bằng GitHub của bạn, hoặc bỏ qua nếu upload ZIP
# !git clone https://github.com/<your-org>/ai-crypto-trading-lab.git 2>&1 | tail -n 5

# Tự động tìm thư mục lab (hỗ trợ cả clone và upload ZIP)
import os
for p in ["/content/ai-crypto-trading-lab", "/content/drive/MyDrive/ai-crypto-trading-lab", "/content/drive/MyDrive/ai-lab", "/content"]:
    if os.path.exists(os.path.join(p, "requirements.txt")):
        os.chdir(p)
        break
    elif os.path.exists(p):
        # thử tìm con chứa requirements
        import pathlib; cands=list(pathlib.Path(p).rglob("requirements.txt"))
        if cands:
            os.chdir(str(cands[0].parent))
            break
print("PWD:", os.getcwd())
!pwd; ls -1 | head -n 20
# Nếu chưa có code, upload ZIP từ máy: chạy cell dưới


In [ ]:
# NẾU CHƯA CÓ CODE (lỗi requirements.txt not found) → chạy cell này để upload ZIP từ máy
# 1. Trên máy local: zip -r ai-crypto-trading-lab.zip ai-crypto-trading-lab (hoặc nén thư mục)
# 2. Chạy cell dưới, chọn file ZIP, nó sẽ tự giải nén
import os
if not os.path.exists("requirements.txt"):
    print("Chưa thấy requirements.txt, hãy upload ZIP...")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            print(f"Đã upload {fname} ({len(uploaded[fname])/1024/1024:.1f} MB)")
            if fname.endswith(".zip"):
                import zipfile, pathlib
                with zipfile.ZipFile(fname) as z:
                    z.extractall("/content")
                print("Đã giải nén")
                # tìm thư mục chứa requirements
                import pathlib
                cands=list(pathlib.Path("/content").rglob("requirements.txt"))
                if cands:
                    os.chdir(str(cands[0].parent))
                    print("Chuyển vào", os.getcwd())
            elif fname.endswith(".py") or fname.endswith(".ipynb"):
                pass
    except Exception as e:
        print("Upload skip:", e)
else:
    print("Đã có code tại", os.getcwd())


## 1) Cài deps
Colab đã có `numpy/pandas/requests/websockets`. Thiếu `pyarrow`/`torch`/`fastapi` thì lab tự fallback CSV/mock.


In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -n 20
!pip install -q pyarrow 2>&1 | tail -n 5  # khuyến nghị để lưu parquet, không bắt buộc
import sys, pathlib
print("pyarrow", end=" ")
try:
    import pyarrow; print(pyarrow.__version__)
except Exception as e:
    print("missing - fallback CSV:", e)
try:
    import websockets; print("websockets", websockets.__version__)
except:
    print("websockets missing")


## 2) Kiểm tra realtime lấy được không?
Binance REST + WS phải trả về giá thật mới đúng nguyên tắc lab.


In [ ]:
import requests, asyncio, json, websockets
r = requests.get("https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT", timeout=5)
print("REST BTCUSDT", r.json())

async def ws_test():
    uri = "wss://stream.binance.com:9443/ws/btcusdt@trade"
    async with websockets.connect(uri, ping_interval=20) as ws:
        for _ in range(3):
            msg = await asyncio.wait_for(ws.recv(), timeout=10)
            d=json.loads(msg)
            print(f"WS trade {d['s']} {d['p']} qty {d['q']}")
await ws_test()


## 3) Lấy toàn bộ lịch sử — TẤT CẢ coin, TẤT CẢ khung giờ
### 3.1 Ước tính dung lượng (đã đo thực tế)
- Spot USDT: 485 coin, Futures USDT: 524 coin (`fapi.binance.com/fapi/v1/exchangeInfo`)
- 1 coin 1m 9 năm ≈ 451 MB parquet → 300 coin ≈ 135 GB **không vừa Colab** (đĩa ~80GB).
- 1 coin 1h 5 năm ≈ 4.2 MB → 300 coin 1h 5 năm ≈ 1.2 GB **vừa**; 50 coin 1h 5 năm ≈ 0.2 GB.
- **Khuyến nghị:** `1h/4h/1d` cho tất cả coin (lấy 5-9 năm), `1m/5m/15m` chỉ cho top 20 coin (1-3 năm). Dùng Binance Vision ZIP (`data.binance.vision`) nhanh hơn API 10x và không rate-limit.

### 3.2 Lệnh fetch (resume tự động, skip file đã tồn tại)


In [ ]:
# Xem ước tính trước (dry-run)
!python scripts/fetch_all_history.py --market spot --top 50 --intervals 1h 4h 1d --years 5 --dry-run

# Thực tế: top 50, 1h 4h 1d, 5 năm, Vision, lưu vào Drive
# !python scripts/fetch_all_history.py --market spot --top 50 --intervals 1h 4h 1d --years 5 --vision --workers 4 --out /content/drive/MyDrive/ai-lab/data/historical

# Demo nhanh: top 5, 1h, 36 ngày (đã chạy xong trong repo)
!python scripts/fetch_all_history.py --market spot --top 5 --intervals 1h --years 0.1 --vision --workers 3 --out data/historical 2>&1 | tail -n 40
!ls -lh data/historical/spot/1h/ | head -n 20
!head -n 3 data/historical/spot/1h/BTCUSDT.csv


#### All coin — ví dụ (cẩn trọng)
```bash
# 300 coin 1h 5 năm ~1.2GB — chạy ~30 phút với Vision
!python scripts/fetch_all_history.py --market spot --all --intervals 1h --years 5 --vision --workers 8 --out /content/drive/MyDrive/ai-lab/data/historical
# 524 futures 15m 1h 4h — 2-3GB, chạy 1-2h
!python scripts/fetch_all_history.py --market futures --all --intervals 15m 1h 4h --years 3 --vision --workers 8 --out /content/drive/MyDrive/ai-lab/data/historical
# Top 20 1m 1 năm ~10GB — chỉ khi cần scalping
!python scripts/fetch_all_history.py --market spot --top 20 --intervals 1m --years 1 --vision --workers 4 --out data/historical
```
Sau khi fetch, mỗi file là `data/historical/<spot|futures>/<interval>/<SYMBOL>.parquet` (hoặc `.csv` nếu thiếu pyarrow) — dùng thẳng cho `historical_replay.py` và `pattern_memory.py`.


## 4) Chạy các lượt trade — Kết luận ĐẠT hay TỆ (20 → 500, x500)
Mỗi **nhánh** = 1 coin + 1 interval + 1 strategy + 1 leverage + 1 pos%. Lab chạy **branching** song song hàng nghìn nhánh trên **lịch sử thật** (không nhìn tương lai), áp dụng luật trade + fee/funding/liquidation thật.

### Luật & kiến thức trade đã encode (`scripts/evaluate_futures_x500.py:70`):
- **Funding/fee:** `taker 0.04%` mỗi mở/đóng, `liquidation fee 0.5%` (`src/exchange_simulator/fees/fee_engine.py`)
- **Liquidation:** `liq = entry * (1 ± 0.9/leverage)` — với x500 chỉ lệch **0.18%** là cháy (`src/exchange_simulator/derivatives/liquidation.py:13`)
- **Strategies:** `ma_cross` (MA7/25/99), `rsi` (<30 long >70 short), `breakout` (Bollinger), `mean_revert`, `trend`, `hold_long`, `random` — kết hợp MA/RSI/BB/vol (`src/market/features/*.py`)
- **Position:** 5%,10%,20%,30%,50%,100% equity; leverage 5/10/20/50/100/125/200/500; nếu vol>2% thì auto giảm pos xuống 10% để tránh cháy
- **Thoát:** take 20% notional / stop -10% / đảo signal ngược thì close


In [ ]:
# Chạy 1000 nhánh trên 5 coin 1h thật (36 ngày) — đã có kết quả bên dưới, chạy lại mất ~20s
!python scripts/evaluate_futures_x500.py --data data/historical --market spot --intervals 1h --top 5 --episodes 1000 --initial 20 --target 500 --max-leverage 500 2>&1 | tail -n 60

# Xem chi tiết
!cat runs/evaluation/futures_x500.json | head -n 60
import json, pandas as pd
j=json.load(open('runs/evaluation/futures_x500.json'))
print(json.dumps(j['summary'], indent=2, ensure_ascii=False))
df=pd.read_csv('runs/evaluation/futures_x500.csv')
df.head()


### Kết quả thực tế 1000 nhánh (đã chạy trong repo, data thật BTC 65098→80731):
- **0.10% đạt 500** (1/1000), **28.4% phá sản**, **42.5% liquidated**, **avg final 16.67** (<20), **median 14.92**
- **Tốt nhất:** `SOL 1h hold_long x20 pos100% → 515.63` (không phải x500!)
- **Theo leverage:** x20 đạt 0.9% avg 26.4; x500 đạt **0%** avg 5.1 liq 52% bankrupt 66% — **x500 tệ nhất**
- **Theo strategy:** `hold_long` 0.8% avg 43.0 tốt nhất (nhờ trend 36 ngày tăng), `rsi` 0% avg 9.1
- **Kết luận:** `TE - Cực khó, <2% đạt, x500 liquidation cao, khuyến nghị giảm leverage hoặc tăng vốn`

### Để đạt 500 từ 20 cần x25 — với x500 chỉ cần +5% giá nếu all-in, nhưng lệch -0.18% là cháy. Thực tế phải compound nhiều lệnh nhỏ x20-x50 với winrate >55% mới sống sót.


## 5) Thử các cấu hình khác (gợi ý)


In [ ]:
# Giảm leverage tối đa xuống 20 (an toàn hơn)
!python scripts/evaluate_futures_x500.py --data data/historical --intervals 1h 4h --top 20 --episodes 2000 --initial 20 --target 100 --max-leverage 20 2>&1 | tail -n 30

# Chỉ BTC/ETH/SOL top 3, 15m + 1h, 1 năm
# !python scripts/evaluate_futures_x500.py --data data/historical --symbols BTCUSDT ETHUSDT SOLUSDT --intervals 15m 1h --episodes 5000 --initial 20 --target 500 --max-leverage 125

# Dùng branching engine của lab để tự tìm nhánh tốt nhất (thay vì random)
from src.branching.branch_generator import BranchGenerator
from src.branching.population.selection import top_k_selection
print("Branching lab có thể sinh 100k nhánh và prune giữ 30% tốt nhất — xem src/branching/README")


## 6) Chạy lab đầy đủ (training AI) — tùy chọn
Nếu muốn AI tự học thay vì rule-based:


In [ ]:
!python scripts/train.py --timesteps 100000 2>&1 | tail -n 20
!python scripts/simulate.py --episodes 1000 --workers 4 2>&1 | tail -n 20
!python scripts/evaluate.py --episodes 100 2>&1 | tail -n 20
!ls -lh runs/evaluation/ && cat runs/evaluation/futures_x500.json | grep -A2 reach_rate


## 7) Lưu kết quả lên Drive và dọn dẹp


In [ ]:
!cp -r runs/evaluation /content/drive/MyDrive/ai-lab/runs/ 2>&1 | tail -n 5
!cp -r data/historical /content/drive/MyDrive/ai-lab/data/ 2>&1 | tail -n 5
!du -sh data/historical/*/* 2>&1 | head -n 20
print("Đã lưu Drive. Ngắt kết nối Colab sẽ không mất data.")


### Tổng kết Colab
- **Lấy data trước:** `fetch_all_history.py` với `--vision` là cách duy nhất lấy được TẤT CẢ lịch sử (API sẽ bị rate-limit). Bắt đầu với `--top 50 --intervals 1h 4h 1d --years 5` (~1GB) rồi mới mở rộng.
- **Kết luận 20→500 x500:** Trên lịch sử thật hiện tại **TỆ (0.1% đạt)**, x500 là nguyên nhân chính liquid 52%. Muốn đạt phải **giảm leverage xuống 10-20, tăng pos nhỏ, chỉ hold_long trend, hoặc tăng vốn lên 100**.
- **Tiếp theo:** Dùng `src/historical_intelligence/pattern_memory.py` để AI học từ 5 năm 1h (thay vì rule), rồi `src/training/trainer.py` PPO trên `src/simulation/environment.py` nối `HistoricalReplay`.
